# 01 — Data: check, labels, samples, packaging

Runs on the local machine. Input: `prep_v1` (read-only). Output: `data/samples_v2` (uploaded to Kaggle) and `data/sealed_v2` (test, never leaves the machine). See the plan sections 3 and 15.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "uc04_v2.json").exists())
PY = sys.executable  # run this notebook with the repo venv (.venv)


def run(script, *args):
    """Run a repo script and stream its output; stop on failure."""
    cmd = [PY, str(REPO / "scripts" / script), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", env={**__import__("os").environ, "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed ({p.returncode})")

## 1. Check prep_v1 (plan 3.1)
19 named checks; test `label_map` is read from the .npy header only.

In [ ]:
run("check_data.py", "--workers", 10)

## 2. Try labels and PPV on 3 train cases

In [ ]:
run("try_labels.py")

## 3. Build samples_v2 (NB01)
**Locked by default.** `samples_v2` is the version checked and uploaded (`samples.json` sha256 `9b17751a…`). Rebuilding changes it: ask the team first, then set `REBUILD_SAMPLES = True`.

In [ ]:
REBUILD_SAMPLES = False
if REBUILD_SAMPLES:
    run("build_samples.py", "--workers", 10, "--force")
else:
    print("skipped: samples_v2 kept as uploaded")

## 4. Package for Kaggle and verify independently
Writes `dist/` only (never touches `samples_v2`). `verify_package.py` lives outside the repo (`D:\SafeAnes\verify_package.py`).

In [ ]:
run("package_for_kaggle.py")
QC = r"D:\SafeAnes\SafeAnes\data\prep_v1\qc.csv"
subprocess.run([PY, r"D:\SafeAneserify_package.py", "--samples", REPO / "data" / "samples_v2", "--qc", QC,
                "--zip", REPO / "dist" / "uc04-samples-v2.zip", "--out", REPO / "dist" / "for_review" / "package_check.json"], check=True)